# 09. Lenguas con pocos recursos: los ojos en náhuatl

En el cuaderno 07 todo venía listo: CLiC nos dio las concordancias de Dickens. En el 08 tuvimos que limpiar los textos, pero spaCy nos dio lemas y categorías gramaticales con una línea de código.

Ahora vamos a intentar lo mismo con el **náhuatl**. Es una de las lenguas indígenas con más hablantes en México, con una tradición escrita de casi cinco siglos. Aun así, en términos computacionales es una lengua de **pocos recursos**:

- No hay modelo de spaCy para el náhuatl.
- Los corpus disponibles son pequeños y mezclan variantes, épocas y ortografías.
- Hay pocas herramientas, y cada una cubre sólo algunas variantes.

El objetivo de este cuaderno no es llegar a un resultado definitivo. Es ver **qué trabajo extra hace falta** y por qué, para entender mejor los retos y todo lo que queda por hacer.

Usaremos **elotl**, una biblioteca de Python de la comunidad mexicana de PLN [Comunidad Elotl](https://elotl.mx). Trae corpus paralelos y herramientas para lenguas de México.

## 0. Preparación

`elotl` no tiene dependencias, así que se instala rápido. FlexiConc se instala igual que en los cuadernos anteriores.

In [ ]:
!apt-get -qq install -y libicu-dev pkg-config > /dev/null
!pip install -q 'flexiconc[notebooks,web,ICU,partitioning,annotation]' elotl

In [ ]:
import re
from collections import Counter

import pandas as pd

import elotl.corpus
from elotl.nahuatl.orthography import Normalizer
import elotl.nahuatl.morphology

from flexiconc import Concordance
from flexiconc.utils.notebook_utils import show_kwic

---
## 1. El corpus Axolotl

`elotl.corpus.list_of_corpus()` muestra los corpus incluidos:

In [ ]:
elotl.corpus.list_of_corpus()

Los tres son **corpus paralelos**: cada texto en lengua indígena viene con su traducción al español. Usaremos **Axolotl** (Gutierrez-Vasques, Sierra y Pompa, 2016). Reúne textos en náhuatl de muchas fuentes: crónicas del siglo XVI, documentos coloniales, cuentos, recetarios, libros de texto y materiales de enseñanza.

`load()` devuelve una lista de filas. Cada fila tiene cinco campos: español, náhuatl, variante, documento y código ISO 639-3 de la variante. La convertimos en un DataFrame.

In [ ]:
ax = pd.DataFrame(elotl.corpus.load("axolotl"),
                  columns=["es", "nah", "variante", "documento", "iso"])
print(len(ax), "pares de oraciones")
ax.sample(5, random_state=3)

### ¿Qué variantes hay?

"Náhuatl" no es una sola variedad. Hay náhuatl clásico (el de los textos coloniales) y muchas variantes actuales, con diferencias de vocabulario, morfología y pronunciación.

In [ ]:
ax["variante"].replace("", "(sin dato)").value_counts()

Más de 2000 pares no dicen de qué variante son. Si quisiéramos comparar variantes, tendríamos que averiguarlo documento por documento. Éste es el primer problema común en lenguas con pocos recursos: los **metadatos** son incompletos.

### ¿Qué tan grande es?

In [ ]:
def palabras(texto):
    return re.findall(r"\w+", texto.lower())

n_nah = sum(len(palabras(t)) for t in ax["nah"])
n_es = sum(len(palabras(t)) for t in ax["es"])
print(f"náhuatl: {n_nah:,} palabras")
print(f"español: {n_es:,} palabras")

Unas 290 000 palabras en náhuatl. Es menos que las cinco novelas del cuaderno 08, y está repartido entre siete variantes y cinco siglos. Como referencia: el corpus con que se entrenó el modelo de spaCy para el español tenía alrededor de medio millón de palabras **anotadas a mano**, y los modelos de lenguaje grandes se entrenan con miles de millones.

Nota también que el español tiene **más** palabras que el náhuatl para decir lo mismo. Ya veremos por qué.

### Limpieza

Algunas oraciones empiezan con números de párrafo o con un punto y un tabulador. Los quitamos de los dos lados.

In [ ]:
print(repr(ax.loc[14855, "nah"]))

def limpiar(texto):
    return re.sub(r"^\s*[\d.]*\s*\t\s*", "", texto).strip()

ax["es"] = ax["es"].map(limpiar)
ax["nah"] = ax["nah"].map(limpiar)
print(repr(ax.loc[14855, "nah"]))

---
## 2. Primer problema: la ortografía

El náhuatl se ha escrito con muchas ortografías distintas. Los frailes del siglo XVI adaptaron la del español de su época. En el siglo XX surgieron otras: la de la SEP, la del INALI, la que usan muchos lingüistas (la de Andrews, Campbell y Karttunen, llamada ACK)... Una misma palabra puede aparecer escrita de varias formas:

| Clásica | SEP | INALI | Significado |
|---|---|---|---|
| *quiza* | *kisa* | *kisa* | 'sale' |
| *huel* | *uel* | *wel* | 'bien, poder' |
| *tlacatl* | *tlakatl* | *tlakatl* | 'persona' |
| *quitzacua* | *kitsakua* | *kitsakwa* | 'lo cierra' |

Para una computadora, *quiza* y *kisa* son dos palabras sin relación. `Normalizer` de elotl convierte cualquier texto a una sola ortografía. Probemos con las tres que ofrece:

In [ ]:
ejemplo = "Auh in yehuatl quitzacua in calli, quiza huel ixquich tlacatl."
for ortografia in ["sep", "inali", "ack"]:
    print(f"{ortografia:6}", Normalizer(ortografia).normalize(ejemplo))

Normalizamos todo el lado náhuatl a la ortografía del **INALI** (la norma oficial). Tarda unos segundos.

In [ ]:
normalizador = Normalizer("inali")
ax["nah_norm"] = ax["nah"].map(normalizador.normalize)
ax[["nah", "nah_norm"]].sample(4, random_state=1)

¿Cuánto ayuda? Contemos cuántas formas distintas (*tipos*) hay antes y después:

In [ ]:
tipos_antes = Counter(p for t in ax["nah"] for p in palabras(t))
tipos_despues = Counter(p for t in ax["nah_norm"] for p in palabras(t))
print("tipos antes: ", len(tipos_antes))
print("tipos después:", len(tipos_despues))

def proporcion_hapax(conteo):
    return sum(1 for n in conteo.values() if n == 1) / len(conteo)

tipos_es = Counter(p for t in ax["es"] for p in palabras(t))
print(f"aparecen una sola vez (náhuatl): {proporcion_hapax(tipos_despues):.0%}")
print(f"aparecen una sola vez (español): {proporcion_hapax(tipos_es):.0%}")

La normalización une casi 3000 formas, pero **dos de cada tres** formas del náhuatl siguen apareciendo una sola vez en todo el corpus. En las traducciones al español, que dicen lo mismo, la proporción es menor a la mitad.

La causa no es la ortografía. Es la morfología del náhuatl, como veremos enseguida.

---
## 3. Segundo problema: ¿qué es una palabra?

En el cuaderno 08 buscamos `[lemma="ojo"]` y listo. Aquí no hay lematizador, así que empecemos por el otro lado: buscamos *ojo* u *ojos* en las **traducciones al español** y vemos qué dice el náhuatl.

In [ ]:
con_ojo = ax[ax["es"].str.contains(r"\bojos?\b", case=False)]
print(len(con_ojo), "oraciones")
pd.set_option("display.max_colwidth", 120)
con_ojo[["es", "nah_norm", "iso"]].head(12)

'Ojo' en náhuatl es *ixtelolotl* (en la Huasteca, *ixtiyolotl*). Veamos cuántas de esas oraciones lo contienen:

In [ ]:
raiz_ojo = r"ixtelol|ixtiyol"
print(con_ojo["nah_norm"].str.contains(raiz_ojo).sum(), "de", len(con_ojo))

Muy pocas. ¿Dónde están los ojos en las demás? Mira estas dos:

- *se le hincharon los ojos* → ***ix**poposawak*
- *y sus ojos se hicieron saltones* → ***ix**polontik elito*

El náhuatl es una lengua **polisintética**. Un verbo puede **incorporar** un sustantivo: *ix-poposawa* es literalmente 'hincharse-de-cara/ojos'. La raíz *ix-* ('cara, ojo, superficie') queda dentro del verbo. No hay una "palabra" *ojo* que buscar.

Esto explica por qué el español tenía más palabras que el náhuatl: lo que el español dice con cinco palabras (*se le hincharon los ojos*) el náhuatl lo puede decir con una.

### ✅ Ejercicio 1 (resuelto)

Busca en `con_ojo` otras palabras en náhuatl que contengan *ix* y anota qué parte de la traducción al español les corresponde.

In [ ]:
for _, fila in con_ojo.iterrows():
    con_ix = [p for p in palabras(fila["nah_norm"]) if "ix" in p]
    if con_ix:
        print(con_ix, "|", fila["es"][:90])

Muchas formas sí tienen la raíz *ix-*:

- *ixtelolohtexohtik*, *ixkafetik*, *ixtliltik*: 'de ojos azules', 'de ojos café', 'de ojos negros'. Una sola palabra para cada adjetivo compuesto.
- *ixpoposawak*, *ixkwatolpoposawak*: 'se le hincharon los ojos', 'se le hincharon los párpados'.
- *ixko*, *amixko*: 'en la cara', 'en su cara (de ustedes)', que el traductor vierte como 'pongo los ojos en vuestro rostro'.
- *teixmimikti*: 'mortificaba los ojos (de la gente)'.

Pero no todo lo que contiene *ix* tiene que ver con los ojos: *kwix* es 'acaso'; *mochixka* viene de *chiya* 'esperar'; *kikixtilia* viene de *kixtia* 'sacar'. Ése es el siguiente problema.

---
## 4. Una concordancia hecha a mano

FlexiConc no tiene una función para cargar Axolotl, pero acepta tres tablas de pandas. Armarlas nosotros nos deja ver qué hay debajo de una concordancia:

- **`metadata`**: una fila por línea de la concordancia. Aquí guardamos la variante, el documento y la **traducción al español**, que nos servirá para interpretar cada línea.
- **`tokens`**: una fila por palabra, con `line_id` (a qué línea pertenece) e `id_in_line` (su posición en la línea).
- **`matches`**: dónde está el nodo de cada línea. `match_start` y `match_end` son posiciones en la tabla `tokens`.

Usamos cada oración completa como contexto. `patron` es una expresión regular que debe coincidir con la palabra **completa**.

In [ ]:
def tokenizar(texto):
    return re.findall(r"\w+|[^\w\s]", texto.lower())

def concordancia(patron, columna="nah_norm"):
    patron = re.compile(patron)
    metadata, tokens, matches = [], [], []
    for _, fila in ax.iterrows():
        palabras_oracion = tokenizar(fila[columna])
        for posicion, palabra in enumerate(palabras_oracion):
            if not patron.fullmatch(palabra):
                continue
            linea = len(metadata)
            inicio = len(tokens)
            metadata.append({"line_id": linea, "variante": fila["variante"] or "(sin dato)",
                             "documento": fila["documento"], "español": fila["es"]})
            for i, p in enumerate(palabras_oracion):
                tokens.append({"line_id": linea, "id_in_line": i, "word": p})
            matches.append({"line_id": linea, "match_start": inicio + posicion,
                            "match_end": inicio + posicion, "slot": 0})
    return Concordance(metadata=pd.DataFrame(metadata),
                       tokens=pd.DataFrame(tokens),
                       matches=pd.DataFrame(matches))

Empezamos con la raíz de 'ojo':

In [ ]:
C_ojo = concordancia(r"\w*(ixtelol|ixtiyol)\w*")
show_kwic(C_ojo.root, metadata_columns=["variante", "español"])

Sólo un puñado de líneas. Con la raíz completa tenemos buena **precisión** (todo lo que encontramos es 'ojo'), pero muy poca **cobertura** (*recall*): perdemos todos los casos con *ix-* incorporado.

### Buscar el morfema *ix-*

Probemos lo contrario: cualquier palabra que contenga *ix* al principio, después de algunos prefijos comunes. La expresión regular `(prefijos)*ix\w*` acepta, por ejemplo, *ix-ko*, *m-ix-ko*, *ki-ix-tlalis*.

In [ ]:
prefijos = r"(?:o|ni|ti|an|ki|k|mo|no|i|to|amo|in|im|te|tla|m)*"
C_ix = concordancia(prefijos + r"ix\w*")
C_ix.root.line_count

Miles de líneas. Agrupamos por la forma del nodo (posición `0`) para ver cuáles son las más frecuentes:

In [ ]:
por_forma = C_ix.root.add_arrangement_node(
    grouping=("Partition by Ngrams", {"positions": [0], "tokens_attribute": "word"}),
    ordering=[])
show_kwic(por_forma, n=3, n_groups=40, metadata_columns=["español"])

### ✅ Ejercicio 2 (resuelto)

Despliega los grupos (haz clic en **▶**) y usa la traducción al español para decidir: ¿cuáles de estas formas tienen realmente la raíz *ix-* 'cara, ojo'?

Algunos casos claros:

| Forma | Análisis | ¿Contiene *ix-* 'cara'? |
|---|---|---|
| *ixpan*, *imixpan*, *teixpan* | (*i-*/*im-*/*te-*) + *ix* + *-pan* 'delante de (él / ellos / la gente)' | **Sí** |
| *ixko*, *mixko* | (*m-*) + *ix* + *-ko* 'en la cara (de ti)' | **Sí** |
| *ixochiyo* | *i-* + *xochi* + *-yo* 'su flor' | No: es el posesivo *i-* + *x* de *xochitl* |
| *ixiwtsin* | *i-* + *xiw* + *-tsin* 'su año, su hoja' | No: posesivo + *xiwitl* |
| *mixtli* | *mix-tli* 'nube' | No |
| *kikixtis* | *ki-kixti-s* 'lo sacará' | No: *kixtia* 'sacar' |

Una expresión regular no puede distinguir *i-xochi-yo* de *ix-ko*. Para eso hace falta **segmentar** las palabras en morfemas, y para segmentar hace falta conocer la lengua: el léxico, los prefijos, las reglas de fusión. La precisión de esta búsqueda es baja.

Nota también que *ix* 'cara' se ha **gramaticalizado**: *ixpan* ya no significa 'en su cara' sino 'delante de, en presencia de'. Algo parecido pasa en español con *frente a* o *de cara a*.

---
## 5. La posesión, otra vez

En el cuaderno 07 vimos *his eyes* y en el 08, *cerró los ojos*. En náhuatl, las partes del cuerpo y los **sustantivos relacionales** como *ixpan* llevan un **prefijo posesivo**:

| Prefijo | Persona | Ejemplo |
|---|---|---|
| *no-* | 1.ª sg. | *noixpan* 'delante de mí' |
| *mo-* | 2.ª sg. | *moixpan* 'delante de ti' |
| *i-* | 3.ª sg. | *iixpan* 'delante de él/ella' |
| *to-* | 1.ª pl. | *toixpan* 'delante de nosotros' |
| *amo-* | 2.ª pl. | *amoixpan* 'delante de ustedes' |
| *in-*, *im-* | 3.ª pl. | *imixpan* 'delante de ellos' |
| *te-* | indefinido humano | *teixpan* 'delante de la gente' |

Hagamos una concordancia de *ixpan* y agreguemos a la tabla `tokens` una columna nueva, **`prefijo`**, calculada con una expresión regular. Es una **anotación hecha por nosotros**, sin modelo, como lo que spaCy hacía por su cuenta en el cuaderno 08.

In [ ]:
C_pan = concordancia(r"(?:no|mo|i|to|amo|in|im|te)?ixpan\w*")

T = C_pan.tokens
T["prefijo"] = ""
nodos = T["offset"] == 0
T.loc[nodos, "prefijo"] = T.loc[nodos, "word"].str.extract(r"^(no|mo|i|to|amo|in|im|te)?ixpan")[0].fillna("(ninguno)")

por_prefijo = C_pan.root.add_arrangement_node(
    grouping=("Partition by Ngrams", {"positions": [0], "tokens_attribute": "prefijo"}),
    ordering=[])
show_kwic(por_prefijo, n=4, metadata_columns=["español"])

FlexiConc acepta la nueva columna como cualquier otro atributo: `tokens_attribute="prefijo"`.

### ✅ Ejercicio 3 (resuelto)

El grupo más grande es "(ninguno)": *ixpan* sin prefijo. Pero un sustantivo relacional **siempre** tiene poseedor. ¿Qué está pasando? Lee las traducciones de ese grupo.

Casi todos son de **3.ª persona**: 'delante de él', y muy a menudo 'ante el juez', 'frente al fuego'. En estos últimos, el poseedor es un sustantivo que aparece **después**: literalmente 'su-delante el juez'.

El prefijo *i-* está ahí, pero se funde con la *i* de *ix*: *i-ixpan* se pronuncia y casi siempre se escribe *ixpan*. Sólo algunos textos escriben *iixpan*.

Nuestra expresión regular no lo sabe. Para anotarlo bien habría que escribir la regla de fusión, revisar los casos y, de nuevo, **saber náhuatl**. Ésta es exactamente la clase de conocimiento que en el español venía "gratis" dentro del modelo de spaCy, porque alguien ya lo había anotado en AnCora.

---
## 6. ¿Y un analizador morfológico?

Sí existe uno: elotl incluye un **analizador morfológico** basado en transductores de estados finitos. Es un conjunto de reglas escritas a mano por lingüistas, no un modelo entrenado. Devuelve el lema, la categoría y los rasgos, al estilo de Universal Dependencies:

In [ ]:
analizador = elotl.nahuatl.morphology.Analyser(lang_code="nci")
for token in analizador.analyse("nimitzitta in tlacatl"):
    print(token.wordform, "→", token.analyses)

*nimitzitta* 'yo te veo' se analiza como el verbo *itta* 'ver', con sujeto de 1.ª persona y objeto de 2.ª. Es justo lo que nos faltaba. Pero sólo cubre tres variantes:

- `nci`: náhuatl clásico
- `azz`: náhuatl de la Sierra de Puebla
- `nhi`: náhuatl de Zacatlán-Ahuacatlán-Tepetzintla (no aparece en Axolotl)

¿Qué tan bien cubre los textos del corpus? Contemos qué porcentaje de palabras recibe al menos un análisis:

In [ ]:
def cobertura(iso, n=300):
    analizador = elotl.nahuatl.morphology.Analyser(lang_code=iso)
    muestra = ax[ax["iso"] == iso].sample(n, random_state=0)
    total = con_analisis = 0
    for texto in muestra["nah"]:
        for token in analizador.analyse(texto):
            if re.fullmatch(r"\w+", token.wordform):
                total += 1
                con_analisis += bool(token.analyses)
    return con_analisis / total

for iso in ["nci", "azz"]:
    print(f"{iso}: {cobertura(iso):.0%} de las palabras con análisis")

La cobertura es mucho mejor para la variante de la Sierra de Puebla que para el náhuatl clásico. El clásico de Axolotl mezcla documentos de muchas épocas y escribanos, con ortografías que el analizador no siempre reconoce. Y para las otras cuatro variantes del corpus no hay analizador.

### ✅ Ejercicio 4 (resuelto)

Prueba el analizador clásico con *ixtelolotl* y con *noixtelolo* ('mi ojo'). ¿Qué pasa?

In [ ]:
for token in analizador.analyse("ixtelolotl noixtelolo"):
    print(token.wordform, "→", token.analyses)

No hay análisis: la palabra 'ojo' no está en el léxico del analizador. Un analizador basado en reglas sólo conoce las raíces que alguien escribió en su diccionario. Agregarlas es trabajo manual, y eso se tiene que hacer para cada variante.

---
## 7. Lo que aprendimos y lo que queda por hacer

Comparemos lo que hicimos en los tres cuadernos:

| | Dickens (07) | Español (08) | Náhuatl (09) |
|---|---|---|---|
| Corpus | listo en CLiC | descargar y limpiar | descargar, limpiar, **normalizar ortografía** |
| Metadatos | completos | los pusimos nosotros | **incompletos** (13 % sin variante) |
| Tokenización | lista | spaCy | expresión regular, pero la "palabra" incluye mucha morfología |
| Lemas y categorías | — | spaCy (con errores) | analizador sólo para 2 de 7 variantes, con huecos en el léxico |
| Buscar 'ojo' | `eyes` | `[lemma="ojo"]` | raíz completa (pocos casos) o *ix-* (muchos falsos positivos) |
| Posesión | posesivo en L1 | artículo en L1, clítico | **prefijo** dentro de la palabra |

En el náhuatl cada paso exigió **conocimiento lingüístico** explícito: las ortografías, los prefijos, la fusión de *i-ixpan*, la incorporación. Por eso el trabajo de lingüistas y hablantes es indispensable en el PLN de lenguas indígenas.

Lo que hace falta, en buena medida, es:

- **Más textos**, de más variantes, con metadatos completos y con permiso y participación de las comunidades hablantes.
- **Corpus anotados a mano** (segmentación morfológica, lemas, categorías) para evaluar herramientas y entrenar modelos.
- **Analizadores y diccionarios** para más variantes.
- **Acuerdos sobre ortografía** o, al menos, normalizadores confiables entre ortografías.

Proyectos como la Comunidad Elotl, Axolotl y los talleres AmericasNLP trabajan en esto. Si te interesa el tema, es un campo donde la formación lingüística es especialmente valiosa.

---
## Referencias

- Andrews, J. R. (2003). *Introduction to Classical Nahuatl* (ed. revisada). University of Oklahoma Press.
- Gutierrez-Vasques, X., Sierra, G. y Pompa, I. H. (2016). Axolotl: a Web Accessible Parallel Corpus for Spanish-Nahuatl. *Proceedings of LREC 2016*, 4210–4214.
- Karttunen, F. (1983). *An Analytical Dictionary of Nahuatl*. University of Texas Press.
- Launey, M. (1992). *Introducción a la lengua y a la literatura náhuatl*. UNAM.
- Mager, M., Gutierrez-Vasques, X., Sierra, G. y Meza-Ruiz, I. (2018). Challenges of language technologies for the indigenous languages of the Americas. *Proceedings of COLING 2018*, 55–69.
- Comunidad Elotl. *py-elotl*. [github.com/ElotlMX/py-elotl](https://github.com/ElotlMX/py-elotl).